# Data Exploration 

**Project:** DS604 – Formula 1 World Championship Database

**Purpose:** Inspect all 14 CSV files before designing the ER diagram, relational schema and DDL.
Nothing is modified or saved to the database here. The output is a set of notes (`02_schema/*.csv`)
used to decide, for every table:

1. Which columns to keep
2. The SQL data type of each column
3. Which columns can be `NULL`
4. The primary key
5. The foreign keys

## 1. Setup

In [1]:
import os
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

DATA_DIR = os.path.join("..", "data", "raw")
OUT_DIR = os.path.join("..", "02_schema")

EXPECTED_FILES = [
    "circuits", "constructors", "drivers", "seasons", "status", "races",
    "results", "sprint_results", "qualifying", "lap_times", "pit_stops",
    "driver_standings", "constructor_standings", "constructor_results",
]

## 2. Load all CSV files

The dataset marks missing values as `\N`, so it is read as `NaN`.
`keep_default_na=False` stops pandas from turning real text such as `"NA"` into missing values.

In [2]:
dfs = {}
for name in EXPECTED_FILES:
    path = os.path.join(DATA_DIR, f"{name}.csv")
    if not os.path.exists(path):
        print(f"MISSING: {path}")
        continue
    dfs[name] = pd.read_csv(path, na_values=["\\N", ""], keep_default_na=False)

print(f"Loaded {len(dfs)} of {len(EXPECTED_FILES)} files")

extra = sorted(f for f in os.listdir(DATA_DIR)
               if f.endswith(".csv") and f[:-4] not in EXPECTED_FILES)
if extra:
    print("Extra files not in the expected list:", extra)

Loaded 14 of 14 files


## 3. Overview: rows, columns and missing values per table

In [3]:
overview = pd.DataFrame([
    {
        "table": name,
        "rows": len(df),
        "columns": df.shape[1],
        "columns_with_nulls": int(df.isnull().any().sum()),
        "total_null_cells": int(df.isnull().sum().sum()),
    }
    for name, df in dfs.items()
])
overview

,table,rows,columns,columns_with_nulls,total_null_cells
0,circuits,77,9,0,0
1,constructors,212,5,0,0
2,drivers,861,9,2,1559
3,seasons,75,2,0,0
4,status,139,2,0,0
5,races,1125,18,11,11349
6,results,26759,18,8,122887
7,sprint_results,360,16,5,73
8,qualifying,10494,9,3,11668
9,lap_times,589081,6,0,0


## 4. First rows of every table

In [4]:
for name, df in dfs.items():
    print(f"\n===== {name} =====")
    display(df.head())


===== circuits =====


,circuitId,circuitRef,name,location,country,lat,lng,alt,url
0,1,albert_park,Albert Park Grand Prix Circuit,Melbourne,Australia,-37.84970,144.96800,10,http://en.wikipedia.org/wiki/Melbourne_Grand_P...
1,2,sepang,Sepang International Circuit,Kuala Lumpur,Malaysia,2.76083,101.73800,18,http://en.wikipedia.org/wiki/Sepang_Internatio...
2,3,bahrain,Bahrain International Circuit,Sakhir,Bahrain,26.03250,50.51060,7,http://en.wikipedia.org/wiki/Bahrain_Internati...
3,4,catalunya,Circuit de Barcelona-Catalunya,Montmeló,Spain,41.57000,2.26111,109,http://en.wikipedia.org/wiki/Circuit_de_Barcel...
4,5,istanbul,Istanbul Park,Istanbul,Turkey,40.95170,29.40500,130,http://en.wikipedia.org/wiki/Istanbul_Park



===== constructors =====


,constructorId,constructorRef,name,nationality,url
0,1,mclaren,McLaren,British,http://en.wikipedia.org/wiki/McLaren
1,2,bmw_sauber,BMW Sauber,German,http://en.wikipedia.org/wiki/BMW_Sauber
2,3,williams,Williams,British,http://en.wikipedia.org/wiki/Williams_Grand_Pr...
3,4,renault,Renault,French,http://en.wikipedia.org/wiki/Renault_in_Formul...
4,5,toro_rosso,Toro Rosso,Italian,http://en.wikipedia.org/wiki/Scuderia_Toro_Rosso



===== drivers =====


,driverId,driverRef,number,code,forename,surname,dob,nationality,url
0,1,hamilton,44.0,HAM,Lewis,Hamilton,1985-01-07,British,http://en.wikipedia.org/wiki/Lewis_Hamilton
1,2,heidfeld,NaN,HEI,Nick,Heidfeld,1977-05-10,German,http://en.wikipedia.org/wiki/Nick_Heidfeld
2,3,rosberg,6.0,ROS,Nico,Rosberg,1985-06-27,German,http://en.wikipedia.org/wiki/Nico_Rosberg
3,4,alonso,14.0,ALO,Fernando,Alonso,1981-07-29,Spanish,http://en.wikipedia.org/wiki/Fernando_Alonso
4,5,kovalainen,NaN,KOV,Heikki,Kovalainen,1981-10-19,Finnish,http://en.wikipedia.org/wiki/Heikki_Kovalainen


... (18 outputs left out)


===== constructor_standings =====


,constructorStandingsId,raceId,constructorId,points,position,positionText,wins
0,1,18,1,14.0,1,1,1
1,2,18,2,8.0,3,3,0
2,3,18,3,9.0,2,2,0
3,4,18,4,5.0,4,4,0
4,5,18,5,2.0,5,5,0



===== constructor_results =====


,constructorResultsId,raceId,constructorId,points,status
0,1,18,1,14.0,NaN
1,2,18,2,8.0,NaN
2,3,18,3,9.0,NaN
3,4,18,4,5.0,NaN
4,5,18,5,2.0,NaN


## 5. Column profile and suggested SQL types

For each column: pandas type, suggested PostgreSQL type, null count, distinct values, maximum text length and a sample value.

Notes on the suggestions:
- A `float64` column whose values are all whole numbers is an integer column with missing values, so it is suggested as `INTEGER`.
- Text columns that look like `YYYY-MM-DD` are suggested as `DATE`, and `HH:MM:SS` as `TIME`.
- Lap and race times such as `1:27.452` stay as `VARCHAR`; the `milliseconds` columns hold the numeric value.
- `pg_column` is the lower-case name PostgreSQL will use for an unquoted identifier.

In [5]:
DATE_PATTERN = r"\d{4}-\d{2}-\d{2}"
TIME_PATTERN = r"\d{2}:\d{2}:\d{2}"
VARCHAR_SIZES = [10, 20, 50, 100, 255, 500]

def suggest_sql_type(series):
    values = series.dropna()
    if values.empty:
        return "VARCHAR(255)  -- all values missing, decide manually"
    if pd.api.types.is_bool_dtype(values):
        return "BOOLEAN"
    if pd.api.types.is_numeric_dtype(values):
        if pd.api.types.is_integer_dtype(values) or (values % 1 == 0).all():
            return "BIGINT" if values.abs().max() > 2_147_483_647 else "INTEGER"
        return "NUMERIC"
    text = values.astype(str)
    if text.str.fullmatch(DATE_PATTERN).all():
        return "DATE"
    if text.str.fullmatch(TIME_PATTERN).all():
        return "TIME"
    longest = int(text.str.len().max())
    for size in VARCHAR_SIZES:
        if longest <= size:
            return f"VARCHAR({size})"
    return "TEXT"

def profile_table(name, df):
    rows = []
    for col in df.columns:
        s = df[col]
        non_null = s.dropna()
        rows.append({
            "table": name,
            "column": col,
            "pg_column": col.lower(),
            "pandas_dtype": str(s.dtype),
            "suggested_sql_type": suggest_sql_type(s),
            "nulls": int(s.isnull().sum()),
            "nullable": "YES" if s.isnull().any() else "NO",
            "distinct_values": int(s.nunique()),
            "max_text_length": int(non_null.astype(str).str.len().max()) if len(non_null) else 0,
            "sample_value": non_null.iloc[0] if len(non_null) else None,
        })
    return pd.DataFrame(rows)

profiles = []
for name, df in dfs.items():
    p = profile_table(name, df)
    profiles.append(p)
    print(f"\n===== {name} ({len(df)} rows) =====")
    display(p.drop(columns="table"))

schema_notes = pd.concat(profiles, ignore_index=True)


===== circuits (77 rows) =====


,column,pg_column,pandas_dtype,suggested_sql_type,nulls,nullable,distinct_values,max_text_length,sample_value
0,circuitId,circuitid,int64,INTEGER,0,NO,77,2,1
1,circuitRef,circuitref,str,VARCHAR(20),0,NO,77,14,albert_park
2,name,name,str,VARCHAR(50),0,NO,77,37,Albert Park Grand Prix Circuit
3,location,location,str,VARCHAR(50),0,NO,75,21,Melbourne
4,country,country,str,VARCHAR(20),0,NO,35,13,Australia
5,lat,lat,float64,NUMERIC,0,NO,77,8,-37.8497
6,lng,lng,float64,NUMERIC,0,NO,77,9,144.968
7,alt,alt,int64,INTEGER,0,NO,66,4,10
8,url,url,str,VARCHAR(100),0,NO,77,71,http://en.wikipedia.org/wiki/Melbourne_Grand_P...



===== constructors (212 rows) =====


,column,pg_column,pandas_dtype,suggested_sql_type,nulls,nullable,distinct_values,max_text_length,sample_value
0,constructorId,constructorid,int64,INTEGER,0,NO,212,3,1
1,constructorRef,constructorref,str,VARCHAR(20),0,NO,212,20,mclaren
2,name,name,str,VARCHAR(50),0,NO,212,25,McLaren
3,nationality,nationality,str,VARCHAR(20),0,NO,24,13,British
4,url,url,str,VARCHAR(100),0,NO,175,73,http://en.wikipedia.org/wiki/McLaren



===== drivers (861 rows) =====


,column,pg_column,pandas_dtype,suggested_sql_type,nulls,nullable,distinct_values,max_text_length,sample_value
0,driverId,driverid,int64,INTEGER,0,NO,861,3,1
1,driverRef,driverref,str,VARCHAR(20),0,NO,861,18,hamilton
2,number,number,float64,INTEGER,802,YES,48,4,44.0
3,code,code,str,VARCHAR(10),757,YES,97,3,HAM
4,forename,forename,str,VARCHAR(20),0,NO,478,17,Lewis
5,surname,surname,str,VARCHAR(50),0,NO,802,23,Hamilton
6,dob,dob,str,DATE,0,NO,843,10,1985-01-07
7,nationality,nationality,str,VARCHAR(20),0,NO,43,17,British
8,url,url,str,VARCHAR(100),0,NO,861,83,http://en.wikipedia.org/wiki/Lewis_Hamilton


... (18 outputs left out)


===== constructor_standings (13391 rows) =====


,column,pg_column,pandas_dtype,suggested_sql_type,nulls,nullable,distinct_values,max_text_length,sample_value
0,constructorStandingsId,constructorstandingsid,int64,INTEGER,0,NO,13391,5,1
1,raceId,raceid,int64,INTEGER,0,NO,1061,4,18
2,constructorId,constructorid,int64,INTEGER,0,NO,160,3,1
3,points,points,float64,NUMERIC,0,NO,579,5,14.0
4,position,position,int64,INTEGER,0,NO,22,2,1
5,positionText,positiontext,str,VARCHAR(10),0,NO,23,2,1
6,wins,wins,int64,INTEGER,0,NO,22,2,1



===== constructor_results (12625 rows) =====


,column,pg_column,pandas_dtype,suggested_sql_type,nulls,nullable,distinct_values,max_text_length,sample_value
0,constructorResultsId,constructorresultsid,int64,INTEGER,0,NO,12625,5,1
1,raceId,raceid,int64,INTEGER,0,NO,1060,4,18
2,constructorId,constructorid,int64,INTEGER,0,NO,175,3,1
3,points,points,float64,NUMERIC,0,NO,61,4,14.0
4,status,status,str,VARCHAR(10),12608,YES,1,1,D


## 6. Primary key check

A key is valid only if it has **0 duplicates** and **0 missing values**.
`lap_times` and `pit_stops` have no ID column, so they use composite keys.

In [6]:
PRIMARY_KEYS = {
    "circuits":              ["circuitId"],
    "constructors":          ["constructorId"],
    "drivers":               ["driverId"],
    "seasons":               ["year"],
    "status":                ["statusId"],
    "races":                 ["raceId"],
    "results":               ["resultId"],
    "sprint_results":        ["resultId"],
    "qualifying":            ["qualifyId"],
    "lap_times":             ["raceId", "driverId", "lap"],
    "pit_stops":             ["raceId", "driverId", "stop"],
    "driver_standings":      ["driverStandingsId"],
    "constructor_standings": ["constructorStandingsId"],
    "constructor_results":   ["constructorResultsId"],
}

pk_rows = []
for table, cols in PRIMARY_KEYS.items():
    if table not in dfs:
        continue
    df = dfs[table]
    missing_cols = [c for c in cols if c not in df.columns]
    if missing_cols:
        pk_rows.append({"table": table, "primary_key": ", ".join(cols),
                        "duplicates": None, "null_keys": None,
                        "valid_pk": f"column not found: {missing_cols}"})
        continue
    duplicates = int(df.duplicated(subset=cols).sum())
    null_keys = int(df[cols].isnull().any(axis=1).sum())
    pk_rows.append({"table": table, "primary_key": ", ".join(cols),
                    "duplicates": duplicates, "null_keys": null_keys,
                    "valid_pk": duplicates == 0 and null_keys == 0})

pk_report = pd.DataFrame(pk_rows)
pk_report

,table,primary_key,duplicates,null_keys,valid_pk
0,circuits,circuitId,0,0,True
1,constructors,constructorId,0,0,True
2,drivers,driverId,0,0,True
3,seasons,year,0,0,True
4,status,statusId,0,0,True
5,races,raceId,0,0,True
6,results,resultId,0,0,True
7,sprint_results,resultId,0,0,True
8,qualifying,qualifyId,0,0,True
9,lap_times,"raceId, driverId, lap",0,0,True


## 7. Foreign key check

For each relationship, counts child rows whose value does not exist in the parent table (orphans).
**0 orphans** means the `FOREIGN KEY` constraint can be added and the data will load without errors.
Missing (`NULL`) foreign key values are allowed and are not counted as orphans.

In [7]:
FOREIGN_KEYS = [
    # (child table, child column, parent table, parent column)
    ("races",                 "year",          "seasons",      "year"),
    ("races",                 "circuitId",     "circuits",     "circuitId"),
    ("results",               "raceId",        "races",        "raceId"),
    ("results",               "driverId",      "drivers",      "driverId"),
    ("results",               "constructorId", "constructors", "constructorId"),
    ("results",               "statusId",      "status",       "statusId"),
    ("sprint_results",        "raceId",        "races",        "raceId"),
    ("sprint_results",        "driverId",      "drivers",      "driverId"),
    ("sprint_results",        "constructorId", "constructors", "constructorId"),
    ("sprint_results",        "statusId",      "status",       "statusId"),
    ("qualifying",            "raceId",        "races",        "raceId"),
    ("qualifying",            "driverId",      "drivers",      "driverId"),
    ("qualifying",            "constructorId", "constructors", "constructorId"),
    ("lap_times",             "raceId",        "races",        "raceId"),
    ("lap_times",             "driverId",      "drivers",      "driverId"),
    ("pit_stops",             "raceId",        "races",        "raceId"),
    ("pit_stops",             "driverId",      "drivers",      "driverId"),
    ("driver_standings",      "raceId",        "races",        "raceId"),
    ("driver_standings",      "driverId",      "drivers",      "driverId"),
    ("constructor_standings", "raceId",        "races",        "raceId"),
    ("constructor_standings", "constructorId", "constructors", "constructorId"),
    ("constructor_results",   "raceId",        "races",        "raceId"),
    ("constructor_results",   "constructorId", "constructors", "constructorId"),
]

fk_rows = []
for child, child_col, parent, parent_col in FOREIGN_KEYS:
    if child not in dfs or parent not in dfs:
        continue
    if child_col not in dfs[child].columns or parent_col not in dfs[parent].columns:
        fk_rows.append({"child": f"{child}.{child_col}", "parent": f"{parent}.{parent_col}",
                        "null_values": None, "orphan_rows": None,
                        "orphan_examples": "column not found", "valid_fk": False})
        continue
    child_values = dfs[child][child_col]
    parent_values = set(dfs[parent][parent_col].dropna())
    non_null = child_values.dropna()
    orphans = non_null[~non_null.isin(parent_values)]
    fk_rows.append({
        "child": f"{child}.{child_col}",
        "parent": f"{parent}.{parent_col}",
        "null_values": int(child_values.isnull().sum()),
        "orphan_rows": len(orphans),
        "orphan_examples": list(orphans.unique()[:5]),
        "valid_fk": len(orphans) == 0,
    })

fk_report = pd.DataFrame(fk_rows)
fk_report

,child,parent,null_values,orphan_rows,orphan_examples,valid_fk
0,races.year,seasons.year,0,0,[],True
1,races.circuitId,circuits.circuitId,0,0,[],True
2,results.raceId,races.raceId,0,0,[],True
3,results.driverId,drivers.driverId,0,0,[],True
4,results.constructorId,constructors.constructorId,0,0,[],True
5,results.statusId,status.statusId,0,0,[],True
6,sprint_results.raceId,races.raceId,0,0,[],True
7,sprint_results.driverId,drivers.driverId,0,0,[],True
8,sprint_results.constructorId,constructors.constructorId,0,0,[],True
9,sprint_results.statusId,status.statusId,0,0,[],True


## 8. Save the notes

In [8]:
os.makedirs(OUT_DIR, exist_ok=True)
schema_notes.to_csv(os.path.join(OUT_DIR, "schema_notes.csv"), index=False)
pk_report.to_csv(os.path.join(OUT_DIR, "primary_key_check.csv"), index=False)
fk_report.to_csv(os.path.join(OUT_DIR, "foreign_key_check.csv"), index=False)
print("Saved to", os.path.abspath(OUT_DIR))

Saved to d:\Daiict 1st sem\DBMS LAB\DBMS PROJECT\DS604-F1-Database-Project\02_schema


## 9. Decisions to record before drawing the ER diagram

Using the outputs above, fill in for every table:

| Decision | Where to look |
|---|---|
| Columns to keep (e.g. drop `url`) | Section 5 |
| SQL type of each column | `suggested_sql_type`, Section 5 (review manually, e.g. `points` should be `NUMERIC` even if all values are whole numbers in one table) |
| `NOT NULL` or nullable | `nullable`, Section 5 |
| Primary key | `valid_pk = True`, Section 6 |
| Foreign keys | `valid_fk = True`, Section 7 |
| Column naming in PostgreSQL | `pg_column`, Section 5 |

These decisions become the ER diagram, the relational schema diagram and the DDL script in `02_schema/`.